# 🧠 MindPulse: Predicting Depression, Anxiety, and Stress (DASS-21)
### Machine Learning & Multi-Task Deep Learning Pipeline
**Author:** Megha Bhandari  
**Tools & Libraries:** Python, Scikit-learn, TensorFlow/Keras, Pandas, NumPy, Matplotlib

---
## 1. Project Overview
This notebook implements an end-to-end clinical psychometric prediction system based on the **Depression, Anxiety, and Stress Scales (DASS-21)** developed by the University of New South Wales (Lovibond & Lovibond).

We evaluate both **classical supervised machine learning algorithms** (Random Forest, Gradient Boosting, Logistic Regression) and a **Multi-Task Deep Neural Network (Keras)** to classify emotional distress into 5 standardized severity tiers (*Normal, Mild, Moderate, Severe, Extremely Severe*).

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

from src.data_loader import generate_synthetic_dass_dataset, SEVERITY_LEVELS
from src.preprocessing import DASSFeatureEngineer

print('All libraries successfully imported!')

## 2. Dataset Synthesis & Clinical Distributions
We generate a psychometrically calibrated dataset (N=3,000) capturing empirical correlations across Depression, Anxiety, and Stress subscales.

In [2]:
df = generate_synthetic_dass_dataset(n_samples=3000, random_state=42)
print('Dataset Shape:', df.shape)
df[['Q1', 'Q2', 'Q3', 'dep_sum', 'anx_sum', 'str_sum', 'Depression_Class']].head()

## 3. Exploratory Data Analysis (EDA) with Matplotlib

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ['dep_sum', 'anx_sum', 'str_sum'], ['Depression Scores', 'Anxiety Scores', 'Stress Scores']):
    ax.hist(df[col], bins=15, color='#0284C7', edgecolor='#0369A1', alpha=0.7)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('DASS-42 Equivalent Score')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Feature Engineering Pipeline (Scikit-learn)
We extract composite distress scores, symptom clusters (autonomic arousal, anhedonia, agitation), and interaction ratios.

In [4]:
feat_eng = DASSFeatureEngineer()
X_raw = df[[f'Q{i+1}' for i in range(21)]]
X_engineered = feat_eng.fit_transform(X_raw)
print('Engineered Feature Matrix Shape:', X_engineered.shape)
print('Total Features:', len(feat_eng.get_feature_names_out()))

## 5. Machine Learning Model Benchmarking
Comparing Random Forest and Gradient Boosting classifiers on stratified test splits.

In [5]:
y_dep = df['Depression_Class']
X_tr, X_te, y_tr, y_te = train_test_split(X_engineered, y_dep, test_size=0.2, random_state=42, stratify=y_dep)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
preds = rf.predict(X_te)
print('Depression Classification Accuracy:', round(accuracy_score(y_te, preds)*100, 2), '%')
print(classification_report(y_te, preds, zero_division=0))